# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
"""
Task type: ranking/scoring the output is an ordered queue.
"""

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
"""
is_declining_label is a predefined label with a threshold I can't prove or defend, and since I'm working on a ranking/scoring problem, I can't use a categorical label anyway — I need a numeric one. 
My target can't be a specific feature; to rank all pages by refresh importance, we need a formula and thresholds that define what makes a page need refreshing.
using multiple features that affect the outcome and that can affect each other. 
Our gate is trend_pct × days_since_last_update — a page only becomes eligible for refresh when its trend is declining and it has had enough time since the last update. 
The features that make up the actual ranking and scoring are impressions_90d, ctr, avg_position, engagement_rate, and scroll_rate. 
These are normalized against content_type, word_count_tier, and client_id, since raw thresholds don't hold globally — a "low" number for one content type or page length is normal for another.
Finally, search_volume and competition act as modifiers that tell the SEO whether a page worth refreshing is also worth chasing.
"""

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
"""
We have no ground truth for the scoring output, so to ensure it's as right as we can, we use is_declining_label as a sanity check. 
We can also print the ranking and manually compare pages to check if they're sorted correctly. 
This holds until we can validate the output against forecasting or put the engine into production and test it against real outcomes.
"""

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [6]:
from IPython.display import display
import numpy as np 
import pandas as pd 

df=pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
engine_features=['impressions_90d','avg_position','ctr','engagement_rate','scroll_rate','trend_pct','days_since_last_update','content_type','word_count','search_volume','competition','main_intent','client_id']
display("One row = one page(one case)")
print(df[engine_features].info())
display(df[engine_features].head(3),df[engine_features].describe())
display(df[['days_since_last_update', 'trend_pct']].head(10))

'One row = one page(one case)'

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   impressions_90d         30000 non-null  int64  
 1   avg_position            30000 non-null  float64
 2   ctr                     30000 non-null  float64
 3   engagement_rate         30000 non-null  float64
 4   scroll_rate             29875 non-null  float64
 5   trend_pct               26612 non-null  float64
 6   days_since_last_update  30000 non-null  int64  
 7   content_type            30000 non-null  str    
 8   word_count              22301 non-null  float64
 9   search_volume           27532 non-null  float64
 10  competition             27532 non-null  float64
 11  main_intent             27626 non-null  str    
 12  client_id               30000 non-null  str    
dtypes: float64(8), int64(2), str(3)
memory usage: 4.2 MB
None


,impressions_90d,avg_position,ctr,engagement_rate,scroll_rate,trend_pct,days_since_last_update,content_type,word_count,search_volume,competition,main_intent,client_id
0,3803,10.6,0.76,5.88,4.55,-41.4,20,keyword article,3221.0,10.0,0.67,transactional,client_f369cb89fc
1,15320,20.3,0.05,0.00,10.00,-57.7,25,keyword article,2481.0,90.0,0.01,informational,client_4e07408562
2,12581,36.5,0.09,0.00,28.57,-60.9,20,keyword article,3515.0,0.0,0.00,informational,client_7f2253d7e2


,impressions_90d,avg_position,ctr,engagement_rate,scroll_rate,trend_pct,days_since_last_update,word_count,search_volume,competition
count,30000.000000,30000.00000,30000.000000,30000.000000,29875.000000,26612.000000,30000.000000,22301.000000,27532.000000,27532.000000
mean,5200.366300,16.34238,0.510733,2.534520,18.212921,-4.785969,46.098300,3107.760325,158.882391,0.146954
std,16838.019547,15.21679,3.279162,8.310096,29.472768,473.861780,42.078709,1452.382598,1518.270825,0.285241
min,1.000000,0.00000,0.000000,0.000000,0.000000,-100.000000,1.000000,8.000000,0.000000,0.000000
25%,81.000000,6.20000,0.000000,0.000000,0.000000,-62.600000,20.000000,2413.000000,0.000000,0.000000
50%,731.000000,10.80000,0.070000,0.000000,5.000000,-33.500000,20.000000,2877.000000,10.000000,0.000000
75%,3615.250000,22.30000,0.290000,1.350000,23.530000,0.000000,104.000000,3666.000000,20.000000,0.130000
max,517715.000000,245.00000,100.000000,100.000000,300.000000,44900.000000,373.000000,9546.000000,74000.000000,1.000000


,days_since_last_update,trend_pct
0,20,-41.4
1,25,-57.7
2,20,-60.9
3,22,-13.8
4,14,-34.7
5,20,-38.9
6,20,-92.3
7,22,0.6
8,20,-58.8
9,104,-29.2


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
"""
The data we have and the problem between our hands can't be solved using one factor. 
We have many factors affecting the outcome — and affecting each other. 
They're bucketed as: the gate factors, which determine which pages even enter the engine; 
the normalizing factors, which account for the fact that not every page has the same conditions and characteristics; 
the scoring factors, which actually score and sort the suspected pages; 
and lastly the modifier factors, giving the SEO a final confirmation of whether a page really needs refreshing, or isn't worth the effort, time, and money.
Encoding all four layers, each with peer-group-relative thresholds, as a single hand-written rule would mean dozens of nested conditions that are hard to tune, hard to explain, and break the moment a new content_type or client is added — that complexity is exactly what a scoring/ranking approach is built to handle.
"""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.